# Pipeline de Unión de *datasets*

El paso final de toda la recolección de datos de la EEA y de *Open-Meteo* consiste en la creación de un *dataset* conjunto por ciudad, donde se cruzan las métricas de calidad del aire con las variables climáticas. Unimos los *datasets* llevando a cabo los siguientes pasos:

1. **Sincronización de series**: dado que los sensores pueden tener ligeros desfases temporales, redondeamos todas las marcas de tiempo a la hora (*dt.floor("h")*) para crear una llave de unión consistente.

2. **Normalización de variables**: renombramos las columnas de valores genéricos a identificadores específicos que incluyen la unidad de medida (ej. PM10 ($ug/m^3$)), permitiendo tener múltiples contaminantes en un solo archivo.

3. **Creación de un *timeline* continuo**: generamos un índice horario ininterrumpido (*pd.date_range*). Esto es fundamental para modelos de series temporales, ya que permite identificar visualmente los periodos de inactividad de los sensores (datos faltantes/*NaN*).

In [ ]:
# ==============================================================================
# IMPORTACIÓN DE LIBRERÍAS
# ==============================================================================

import pandas as pd
from pathlib import Path
import os
import requests
import re
import time

import warnings
warnings.filterwarnings("ignore", category=UserWarning)

In [8]:
# ==============================================================================
# CONFIGURACIÓN DE RUTAS
# ==============================================================================

# La carpeta "datasets" está un nivel por encima de la carpeta actual del proyecto
BASE_PATH = Path("..", "..") / "datasets"

# Carpetas de entrada y salida
FOLDER_POLLUTANTS = BASE_PATH / "archivos_csv" / "ES"
FOLDER_CLIMA      = BASE_PATH / "archivos_clima"
FOLDER_FINAL      = BASE_PATH / "archivos_cont_clima"

# Creamos la carpeta final si no existe
FOLDER_FINAL.mkdir(parents=True, exist_ok=True)

print(f"📂 Carpeta de contaminantes: {FOLDER_POLLUTANTS}")
print(f"📂 Carpeta de clima: {FOLDER_CLIMA}")
print(f"📂 Carpeta final preparada en: {FOLDER_FINAL}")

📂 Carpeta de contaminantes: ..\..\datasets\archivos_csv\ES
📂 Carpeta de clima: ..\..\datasets\archivos_clima
📂 Carpeta final preparada en: ..\..\datasets\archivos_cont_clima


In [9]:
# ==============================================================================
# DICCIONARIO MANUAL DE CONTAMINANTES
# ==============================================================================
# Diccionario manual que traduce los IDs numéricos de contaminantes
# al nombre químico correspondiente.

DICCIONARIO_MANUAL = {
"1": "SO2",
"5": "PM10",
"7": "O3",
"8": "NO",
"9": "NO2",
"10": "CO",
"20": "C6H6",
"38": "NOx",
"6001": "NO2"
}

In [10]:
# ==============================================================================
# ITERACIÓN POR CIUDADES
# ==============================================================================
# Listamos las subcarpetas existentes dentro del directorio de contaminantes.
# Cada subcarpeta corresponde a una ciudad (Madrid, Barcelona, etc.).

ciudades = [f.name for f in FOLDER_POLLUTANTS.iterdir() if f.is_dir()]

for ciudad in ciudades:
    folder_poll = FOLDER_POLLUTANTS / ciudad
    folder_clima = FOLDER_CLIMA / ciudad

    # --------------------------------------------------------------------------
    # PROCESAMIENTO DE CONTAMINANTES
    # --------------------------------------------------------------------------
    dfs_poll = []

    # Buscamos todos los archivos CSV de la ciudad (previamente unificados)
    for f in folder_poll.glob("*.csv"):
        df = pd.read_csv(f)
        if df.empty:
            continue

        # ------------------------------------------------------------------
        # NORMALIZACIÓN TEMPORAL
        # ------------------------------------------------------------------
        # Convertimos las columnas de tiempo al formato Datetime

        df["Start"] = pd.to_datetime(df["Start"], errors="coerce")
        df["End"]   = pd.to_datetime(df["End"], errors="coerce")

        # Generamos una llave temporal redondeada a la hora.
        # Esta columna permitirá unir datos climáticos posteriormente.
        df["Start_hour"] = df["Start"].dt.floor("h")

        # ------------------------------------------------------------------
        # TRADUCCIÓN MANUAL DEL CONTAMINANTE
        # ------------------------------------------------------------------

        pollutant_id = str(df["Pollutant"].iloc[0]).strip()
        unit = str(df["Unit"].iloc[0]).strip()

        # Buscamos el nombre químico en el diccionario manual
        nombre_quimico = DICCIONARIO_MANUAL.get(pollutant_id, pollutant_id)

        # Renombramos la columna "Value" con el nombre químico y su unidad
        col_final = f"{nombre_quimico} ({unit})"

        df = df.rename(columns={"Value": col_final})

        # Guardamos solo las columnas necesarias para el merge posterior
        dfs_poll.append(df[["Start", "End", "Start_hour", col_final]])

    if not dfs_poll:
        continue


    # --------------------------------------------------------------------------
    # UNIÓN DE TODOS LOS CONTAMINANTES
    # --------------------------------------------------------------------------
    # Fusionamos todos los contaminantes de la ciudad en una única tabla.

    df_poll = dfs_poll[0]

    for df_next in dfs_poll[1:]:
        df_poll = pd.merge(
            df_poll,
            df_next,
            on=["Start", "End", "Start_hour"],
            how="outer"
        )

    # --------------------------------------------------------------------------
    # PROCESAMIENTO DE DATOS CLIMÁTICOS
    # --------------------------------------------------------------------------

    dfs_clima = []

    for f in folder_clima.glob("*.csv"):

        df_c = pd.read_csv(f)

        # Conversión del formato de tiempo de Open-Meteo
        df_c["Start_hour"] = pd.to_datetime(
            df_c["time"],
            format="%Y-%m-%dT%H:%M",
            errors='coerce'
        )

        df_c = df_c.drop(columns=["time"])
        dfs_clima.append(df_c)

    if dfs_clima:

        df_clima = dfs_clima[0]

        for df_next in dfs_clima[1:]:
            df_clima = pd.merge(
                df_clima,
                df_next,
                on="Start_hour",
                how="outer"
            )

    else:
        df_clima = pd.DataFrame()


    # --------------------------------------------------------------------------
    # CONSTRUCCIÓN DEL DATASET TEMPORAL FINAL
    # --------------------------------------------------------------------------
    # Generamos un eje temporal continuo para evitar huecos en la serie.

    min_start = df_poll["Start"].min()
    max_end = df_poll["End"].max()

    if pd.isna(min_start) or pd.isna(max_end):
        continue

    idx = pd.date_range(
        start=min_start.floor("h"),
        end=max_end.ceil("h"),
        freq="h"
    )

    df_final = pd.DataFrame({"Start_hour": idx})

    # --------------------------------------------------------------------------
    # INTEGRACIÓN CONTAMINANTES + CLIMA
    # --------------------------------------------------------------------------

    df_final = pd.merge(df_final, df_poll, on="Start_hour", how="left")

    if not df_clima.empty:
        df_final = pd.merge(df_final, df_clima, on="Start_hour", how="left")


    # --------------------------------------------------------------------------
    # PROMEDIO SEGURO DE CONTAMINANTES DUPLICADOS
    # --------------------------------------------------------------------------

    # Eliminamos sufijos generados por merge (_x, _y)
    df_final.columns = [c.replace('_x', '').replace('_y', '') for c in df_final.columns]

    # Guardamos columnas de fecha
    cols_fechas = ["Start_hour", "Start", "End"]
    df_fechas = df_final[cols_fechas].copy()

    # Seleccionamos solo columnas numéricas
    df_numerico = df_final.select_dtypes(include=['number'])

    # Promediamos columnas duplicadas
    df_promediado = df_numerico.T.groupby(level=0).mean().T

    # Unimos nuevamente fechas con datos
    df_final = pd.concat([df_fechas, df_promediado], axis=1)

    # Eliminamos posibles duplicados
    df_final = df_final.loc[:, ~df_final.columns.duplicated()]

    # --------------------------------------------------------------------------
    # RECONSTRUCCIÓN DE COLUMNAS TEMPORALES
    # --------------------------------------------------------------------------

    df_final["Start"] = df_final["Start_hour"]
    df_final["End"]   = df_final["Start_hour"] + pd.Timedelta(hours=1)

    # Redondeo de valores numéricos
    df_final = df_final.round(2)

    # --------------------------------------------------------------------------
    # ORDENACIÓN FINAL DE COLUMNAS
    # --------------------------------------------------------------------------

    df_final = df_final.drop(columns=["Start_hour"])

    cols = ["Start", "End"] + [
        c for c in df_final.columns
        if c not in ["Start", "End"]
    ]

    df_final = df_final[cols]

    # --------------------------------------------------------------------------
    # GUARDADO DEL DATASET FINAL
    # --------------------------------------------------------------------------

    output_file = FOLDER_FINAL / f"{ciudad}.csv"
    df_final.to_csv(output_file, index=False, encoding="utf-8-sig")

print(f"\n🚀 PROCESO FINALIZADO. Los datasets están listos en {FOLDER_FINAL}.")


🚀 PROCESO FINALIZADO. Los datasets están listos en ..\..\datasets\archivos_cont_clima.
